
# EXP01B — Single-Season vs Super-League Berrar Ablation

This is the **final P1/data-side experiment**.

It isolates one question that EXP01 did not isolate:

> Does carrying same-league history across season boundaries improve Berrar's representation, compared with resetting team history at the start of EPL 2015/16?

## Controlled comparison

**Single-season arm**
- StatsBomb EPL 2015/16 target matches only.
- Team history resets at the start of the season.
- Same six Berrar `total` features.
- Same frozen recency `n=28`.
- Same minimum-six eligibility rule.

**Super-League arm**
- Existing TA-approved P1 output.
- Football-Data EPL 2000/01–2014/15 historical context plus causal completed StatsBomb EPL 2015/16 matches.
- Same six Berrar `total` features.
- Same frozen recency `n=28`.

The experiment measures **two different effects**:

1. **Coverage / cold-start reduction** — how many target matches become eligible.
2. **Predictive quality** — on the exact intersection of matches eligible under both arms, which representation gives lower RPS.

No recency value is re-selected. No feature definition is changed. The fixed Random Forest probe is loaded from EXP01's pre-test freeze record. Test results are report-only.


## 1. Configuration

In [1]:

from pathlib import Path

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/ml_project/code/data_pipeline")

CODE_ROOT = DRIVE_PROJECT_ROOT / "extracted" / "md1_data_pipeline_code"
P1_RUN = (
    CODE_ROOT
    / "colab_results"
    / "EPL_2000_01_2015_16_Berrar_P1_HYBRID_REAL"
)
EXP01_RUN = DRIVE_PROJECT_ROOT / "results" / "EXP01_P1_REPRESENTATION_TASK_C"

EXPERIMENT_ROOT = DRIVE_PROJECT_ROOT / "results" / "EXP01B_SUPERLEAGUE_ABLATION"

EXPECTED_SELECTED_N = 28
RESET_EXPERIMENT_OUTPUT = True
RUN_TEST_STAGE = True

# Diagnostic only. These folds use the common TRAIN population and never alter n,
# the feature definitions, or the fixed RF probe.
TEMPORAL_FOLDS = 3


## 2. Mount Drive and validate frozen inputs

In [2]:

import os
import sys
import shutil
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

P1_TARGET = P1_RUN / "data" / "silver" / "p1_target_statsbomb_matches.parquet"
P1_SUPER = P1_RUN / "data" / "gold" / "p1_features_total_selected.parquet"
P1_SPLIT = P1_RUN / "data" / "gold" / "p1_split_manifest.csv"
P1_SUMMARY = P1_RUN / "audit" / "p1_run_summary.json"
P1_SELECTED = P1_RUN / "audit" / "p1_selected_recency.json"
P1_READINESS = P1_RUN / "audit" / "p1_readiness.csv"
P1_LEAKAGE = P1_RUN / "audit" / "p1_leakage_tests.csv"

EXP01_FREEZE = EXP01_RUN / "audit" / "FROZEN_BEFORE_TEST.json"
EXP01_READINESS = EXP01_RUN / "audit" / "exp01_readiness.csv"

REQUIRED_FILES = {
    "p1_berrar_module": CODE_ROOT / "md1_data_pipeline" / "p1_berrar.py",
    "workflow_module": CODE_ROOT / "md1_data_pipeline" / "workflow.py",
    "requirements": CODE_ROOT / "requirements_colab.txt",
    "p1_target_statsbomb": P1_TARGET,
    "p1_superleague_selected": P1_SUPER,
    "p1_split": P1_SPLIT,
    "p1_summary": P1_SUMMARY,
    "p1_selected_recency": P1_SELECTED,
    "p1_readiness": P1_READINESS,
    "p1_leakage": P1_LEAKAGE,
    "exp01_freeze": EXP01_FREEZE,
    "exp01_readiness": EXP01_READINESS,
}

missing = {k: str(v) for k, v in REQUIRED_FILES.items() if not v.exists()}
if missing:
    raise FileNotFoundError(
        "EXP01B cannot start because frozen upstream artifacts are missing:\n"
        + "\n".join(f"  {k}: {v}" for k, v in missing.items())
    )

if RESET_EXPERIMENT_OUTPUT and EXPERIMENT_ROOT.exists():
    shutil.rmtree(EXPERIMENT_ROOT)

for sub in ("data", "models", "predictions", "metrics", "audit", "plots"):
    (EXPERIMENT_ROOT / sub).mkdir(parents=True, exist_ok=True)

print("Canonical P1 run:", P1_RUN)
print("EXP01 run:", EXP01_RUN)
print("EXP01B output:", EXPERIMENT_ROOT)
print("Frozen upstream files detected: PASS")


Mounted at /content/drive
Canonical P1 run: /content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipeline_code/colab_results/EPL_2000_01_2015_16_Berrar_P1_HYBRID_REAL
EXP01 run: /content/drive/MyDrive/ml_project/code/data_pipeline/results/EXP01_P1_REPRESENTATION_TASK_C
EXP01B output: /content/drive/MyDrive/ml_project/code/data_pipeline/results/EXP01B_SUPERLEAGUE_ABLATION
Frozen upstream files detected: PASS


## 3. Install dependencies and import the canonical P1 engine

In [3]:

import subprocess
import py_compile
import importlib

requirements_path = CODE_ROOT / "requirements_colab.txt"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade-strategy",
    "only-if-needed",
    "-r",
    str(requirements_path),
])

for module_name in ("p1_berrar.py", "workflow.py"):
    py_compile.compile(
        str(CODE_ROOT / "md1_data_pipeline" / module_name),
        doraise=True,
    )

CODE_ROOT_STR = str(CODE_ROOT)
sys.path = [p for p in sys.path if p != CODE_ROOT_STR]
sys.path.insert(0, CODE_ROOT_STR)

for module_name in list(sys.modules):
    if module_name == "md1_data_pipeline" or module_name.startswith("md1_data_pipeline."):
        del sys.modules[module_name]

from md1_data_pipeline.p1_berrar import (
    PAPER_TOTAL_FEATURES,
    P1HistoryIndex,
    assert_no_p1_prematch_leakage,
    build_p1_current_team_sets,
    build_p1_features_for_n,
    build_p1_team_match_facts,
)
from md1_data_pipeline.workflow import build_p1_default_config

assert len(PAPER_TOTAL_FEATURES) == 6
print("Canonical Berrar engine imported from:", CODE_ROOT)
print("Paper total feature contract:", list(PAPER_TOTAL_FEATURES))
print("Import/compile validation: PASS")


Canonical Berrar engine imported from: /content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipeline_code
Paper total feature contract: ['p1_home_scr_total', 'p1_home_con_total', 'p1_home_rank_total', 'p1_away_scr_total', 'p1_away_con_total', 'p1_away_rank_total']
Import/compile validation: PASS


## 4. Experiment helpers

In [4]:

from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Mapping, Sequence
import hashlib
import json
import platform
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.model_selection import TimeSeriesSplit

CLASS_ORDER = ("H", "D", "A")


@dataclass(frozen=True)
class ProbeConfig:
    n_estimators: int = 500
    max_depth: int | None = None
    min_samples_leaf: int = 3
    max_features: str = "sqrt"
    class_weight: str | None = None
    random_state: int = 42
    n_jobs: int = -1
    imputer_strategy: str = "median"
    ece_bins: int = 10
    bootstrap_resamples: int = 10000
    temporal_splits: int = 3


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")


def as_bool_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s.dtype):
        return s.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(s.dtype):
        return s.fillna(0).astype(float).ne(0)
    return s.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})


def feature_values_equal(a: pd.Series, b: pd.Series, cols: Sequence[str], atol: float = 1e-12) -> bool:
    for c in cols:
        av, bv = a[c], b[c]
        if pd.isna(av) and pd.isna(bv):
            continue
        try:
            if not np.isclose(float(av), float(bv), equal_nan=True, atol=atol, rtol=0.0):
                return False
        except Exception:
            if str(av) != str(bv):
                return False
    return True


def one_hot(y: Sequence[str], class_order: Sequence[str] = CLASS_ORDER) -> np.ndarray:
    labels = list(y)
    idx = {c: i for i, c in enumerate(class_order)}
    arr = np.zeros((len(labels), len(class_order)), dtype=float)
    for r, label in enumerate(labels):
        if label not in idx:
            raise ValueError(f"Unknown class {label!r}")
        arr[r, idx[label]] = 1.0
    return arr


def per_match_rps(y_true: Sequence[str], probs: np.ndarray, class_order: Sequence[str] = CLASS_ORDER) -> np.ndarray:
    y = one_hot(y_true, class_order)
    probs = np.asarray(probs, dtype=float)
    if probs.shape != y.shape:
        raise ValueError(f"probability shape {probs.shape} != target shape {y.shape}")
    if not np.all(np.isfinite(probs)):
        raise AssertionError("Non-finite probabilities")
    if not np.allclose(probs.sum(axis=1), 1.0, atol=1e-8):
        raise AssertionError("Probabilities do not sum to one")
    cum_p = np.cumsum(probs, axis=1)[:, :-1]
    cum_y = np.cumsum(y, axis=1)[:, :-1]
    return np.sum((cum_p - cum_y) ** 2, axis=1) / (len(class_order) - 1)


def multiclass_brier(y_true: Sequence[str], probs: np.ndarray, class_order: Sequence[str] = CLASS_ORDER) -> float:
    y = one_hot(y_true, class_order)
    return float(np.mean(np.sum((np.asarray(probs) - y) ** 2, axis=1)))


def top_label_ece(y_true: Sequence[str], probs: np.ndarray, class_order: Sequence[str] = CLASS_ORDER, n_bins: int = 10) -> float:
    y_true = np.asarray(list(y_true), dtype=object)
    probs = np.asarray(probs, dtype=float)
    pred_idx = np.argmax(probs, axis=1)
    pred = np.asarray([class_order[i] for i in pred_idx], dtype=object)
    conf = probs.max(axis=1)
    correct = (pred == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i+1]
        mask = (conf >= lo) & ((conf <= hi) if i == n_bins - 1 else (conf < hi))
        if not mask.any():
            continue
        ece += (mask.sum() / len(conf)) * abs(float(correct[mask].mean()) - float(conf[mask].mean()))
    return float(ece)


def metrics_row(name: str, y_true: Sequence[str], probs: np.ndarray, cfg: ProbeConfig) -> dict[str, Any]:
    labels = list(y_true)
    probs = np.asarray(probs, dtype=float)
    idx = np.asarray([CLASS_ORDER.index(y) for y in labels], dtype=int)
    pred = np.asarray(CLASS_ORDER, dtype=object)[np.argmax(probs, axis=1)]
    return {
        "representation": name,
        "n_matches": len(labels),
        "rps": float(per_match_rps(labels, probs).mean()),
        "log_loss": float(-np.mean(np.log(np.clip(probs[np.arange(len(labels)), idx], 1e-15, 1.0)))),
        "brier": multiclass_brier(labels, probs),
        "accuracy": float(accuracy_score(labels, pred)),
        "ece": top_label_ece(labels, probs, n_bins=cfg.ece_bins),
    }


def make_rf_pipeline(cfg: ProbeConfig) -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy=cfg.imputer_strategy)),
        ("rf", RandomForestClassifier(
            n_estimators=cfg.n_estimators,
            max_depth=cfg.max_depth,
            min_samples_leaf=cfg.min_samples_leaf,
            max_features=cfg.max_features,
            class_weight=cfg.class_weight,
            random_state=cfg.random_state,
            n_jobs=cfg.n_jobs,
        )),
    ])


def aligned_predict_proba(model: Pipeline, X: pd.DataFrame) -> np.ndarray:
    raw = model.predict_proba(X)
    learned = [str(c) for c in model.named_steps["rf"].classes_]
    missing = [c for c in CLASS_ORDER if c not in learned]
    if missing:
        raise AssertionError(f"Training split missing classes: {missing}; learned={learned}")
    out = np.column_stack([raw[:, learned.index(c)] for c in CLASS_ORDER])
    if not np.allclose(out.sum(axis=1), 1.0, atol=1e-10):
        raise AssertionError("Predicted probabilities do not sum to one")
    return out


def prediction_frame(name: str, frame: pd.DataFrame, probs: np.ndarray) -> pd.DataFrame:
    out = frame[["match_id", "split", "outcome"]].copy().reset_index(drop=True)
    out["representation"] = name
    out["p_home"] = probs[:, 0]
    out["p_draw"] = probs[:, 1]
    out["p_away"] = probs[:, 2]
    out["predicted_outcome"] = np.asarray(CLASS_ORDER, dtype=object)[np.argmax(probs, axis=1)]
    out["per_match_rps"] = per_match_rps(out["outcome"].astype(str).tolist(), probs)
    return out


def build_coverage_audit(single: pd.DataFrame, superleague: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    req = {"match_id","split","kickoff","p1_eligible","home_total_history_available","away_total_history_available"}
    for label, frame in (("single", single),("superleague", superleague)):
        missing = sorted(req - set(frame.columns))
        if missing:
            raise KeyError(f"{label} frame missing coverage columns: {missing}")
        if frame["match_id"].duplicated().any():
            raise AssertionError(f"{label} has duplicate match_id")

    m = single[list(req)].merge(
        superleague[list(req)], on="match_id", how="outer", suffixes=("_single","_super"), validate="one_to_one", indicator=True
    )
    if not (m["_merge"] == "both").all():
        raise AssertionError("Single-season and Super-League target populations differ")
    # identity fields should agree
    if not (m["split_single"].astype(str) == m["split_super"].astype(str)).all():
        raise AssertionError("Split disagreement between single-season and Super-League features")
    if not (pd.to_datetime(m["kickoff_single"]) == pd.to_datetime(m["kickoff_super"])).all():
        raise AssertionError("Kickoff disagreement between feature arms")

    m["single_eligible"] = as_bool_series(m["p1_eligible_single"])
    m["superleague_eligible"] = as_bool_series(m["p1_eligible_super"])
    m["common_eligible"] = m["single_eligible"] & m["superleague_eligible"]
    m["coverage_gain_superleague"] = m["superleague_eligible"].astype(int) - m["single_eligible"].astype(int)
    m["split"] = m["split_single"].astype(str)
    m["kickoff"] = pd.to_datetime(m["kickoff_single"])

    rows = []
    for split in ["all","train","validation","test"]:
        part = m if split == "all" else m[m["split"] == split]
        rows.append({
            "split": split,
            "total_matches": int(len(part)),
            "single_eligible": int(part["single_eligible"].sum()),
            "superleague_eligible": int(part["superleague_eligible"].sum()),
            "common_eligible": int(part["common_eligible"].sum()),
            "superleague_additional_eligible": int((part["superleague_eligible"] & ~part["single_eligible"]).sum()),
            "single_coverage_pct": float(100 * part["single_eligible"].mean()) if len(part) else np.nan,
            "superleague_coverage_pct": float(100 * part["superleague_eligible"].mean()) if len(part) else np.nan,
        })
    return m, pd.DataFrame(rows)


def build_common_feature_tables(single: pd.DataFrame, superleague: pd.DataFrame, feature_cols: Sequence[str]) -> dict[str, pd.DataFrame]:
    common_ids = set(single.loc[as_bool_series(single["p1_eligible"]), "match_id"].astype(int)) & set(
        superleague.loc[as_bool_series(superleague["p1_eligible"]), "match_id"].astype(int)
    )
    if not common_ids:
        raise AssertionError("No common eligible matches")

    meta_cols = ["match_id","split","kickoff","home_team_id","away_team_id","outcome","goal_margin_raw"]
    tables = {}
    for name, src in (("SINGLE_SEASON", single),("SUPER_LEAGUE", superleague)):
        missing = [c for c in [*meta_cols,*feature_cols] if c not in src.columns]
        if missing:
            raise KeyError(f"{name} missing required columns: {missing}")
        t = src[src["match_id"].astype(int).isin(common_ids)][[*meta_cols,*feature_cols]].copy()
        t = t.sort_values(["kickoff","match_id"]).reset_index(drop=True)
        if t["match_id"].duplicated().any():
            raise AssertionError(f"{name} duplicate common match_id")
        tables[name] = t
    a,b = tables["SINGLE_SEASON"],tables["SUPER_LEAGUE"]
    if a["match_id"].astype(int).tolist() != b["match_id"].astype(int).tolist():
        raise AssertionError("Common match order differs")
    if a["outcome"].astype(str).tolist() != b["outcome"].astype(str).tolist():
        raise AssertionError("Common labels differ")
    if a["split"].astype(str).tolist() != b["split"].astype(str).tolist():
        raise AssertionError("Common split differs")
    return tables


def fit_pair(tables: Mapping[str,pd.DataFrame], feature_cols: Sequence[str], cfg: ProbeConfig, models_dir: Path | None = None) -> dict[str,Pipeline]:
    models={}
    for name in ("SINGLE_SEASON","SUPER_LEAGUE"):
        train=tables[name][tables[name]["split"]=="train"].copy()
        if train.empty:
            raise AssertionError(f"{name} common train is empty")
        model=make_rf_pipeline(cfg)
        model.fit(train[list(feature_cols)],train["outcome"].astype(str))
        if set(map(str,model.named_steps["rf"].classes_)) != set(CLASS_ORDER):
            raise AssertionError(f"{name} common train does not contain all classes")
        models[name]=model
        if models_dir:
            models_dir.mkdir(parents=True,exist_ok=True)
            joblib.dump(model,models_dir/f"{name}.joblib")
    return models


def evaluate_pair(models: Mapping[str,Pipeline], tables: Mapping[str,pd.DataFrame], feature_cols: Sequence[str], cfg: ProbeConfig, split: str):
    preds=[]; metrics=[]
    ids_ref=None
    for name in ("SINGLE_SEASON","SUPER_LEAGUE"):
        part=tables[name][tables[name]["split"]==split].sort_values(["kickoff","match_id"]).reset_index(drop=True)
        if part.empty:
            raise AssertionError(f"No common {split} rows")
        ids=part["match_id"].astype(int).tolist()
        if ids_ref is None: ids_ref=ids
        elif ids != ids_ref: raise AssertionError(f"{split} populations differ")
        probs=aligned_predict_proba(models[name],part[list(feature_cols)])
        preds.append(prediction_frame(name,part,probs))
        metrics.append({**metrics_row(name,part["outcome"].astype(str),probs,cfg),"feature_count":len(feature_cols),"model":"RandomForest fixed EXP01 probe"})
    return pd.concat(preds,ignore_index=True),pd.DataFrame(metrics).sort_values("rps").reset_index(drop=True)


def paired_rps_bootstrap(pred_long: pd.DataFrame, left: str="SUPER_LEAGUE", right: str="SINGLE_SEASON", n_resamples: int=10000, seed: int=42) -> pd.DataFrame:
    l=pred_long[pred_long["representation"]==left][["match_id","per_match_rps"]].rename(columns={"per_match_rps":"left_rps"})
    r=pred_long[pred_long["representation"]==right][["match_id","per_match_rps"]].rename(columns={"per_match_rps":"right_rps"})
    m=l.merge(r,on="match_id",validate="one_to_one").sort_values("match_id")
    if len(m)!=len(l) or len(m)!=len(r) or m.empty:
        raise AssertionError("Bootstrap comparison populations differ")
    d=(m["left_rps"]-m["right_rps"]).to_numpy(float)
    rng=np.random.default_rng(seed)
    means=[]
    remaining=n_resamples
    while remaining:
        b=min(2000,remaining)
        idx=rng.integers(0,len(d),size=(b,len(d)))
        means.append(d[idx].mean(axis=1))
        remaining-=b
    boot=np.concatenate(means)
    return pd.DataFrame([{
        "left":left,"right":right,"n_matches":len(d),
        "delta_definition":"RPS(SUPER_LEAGUE) - RPS(SINGLE_SEASON); negative favors Super League",
        "mean_delta_rps":float(d.mean()),
        "ci95_low":float(np.quantile(boot,0.025)),
        "ci95_high":float(np.quantile(boot,0.975)),
        "bootstrap_resamples":int(n_resamples),
        "bootstrap_seed":int(seed),
        "fraction_bootstrap_delta_lt_0":float(np.mean(boot<0)),
    }])


def temporal_fold_probe(tables: Mapping[str,pd.DataFrame], feature_cols: Sequence[str], cfg: ProbeConfig) -> pd.DataFrame:
    ref=tables["SINGLE_SEASON"][tables["SINGLE_SEASON"]["split"]=="train"].sort_values(["kickoff","match_id"]).reset_index(drop=True)
    other=tables["SUPER_LEAGUE"][tables["SUPER_LEAGUE"]["split"]=="train"].sort_values(["kickoff","match_id"]).reset_index(drop=True)
    if ref["match_id"].astype(int).tolist()!=other["match_id"].astype(int).tolist():
        raise AssertionError("Temporal-fold training populations differ")
    if len(ref) < cfg.temporal_splits + 3:
        raise AssertionError("Too few common training rows for temporal folds")
    splitter=TimeSeriesSplit(n_splits=cfg.temporal_splits)
    rows=[]
    for fold,(tr_idx,va_idx) in enumerate(splitter.split(ref),start=1):
        for name,frame in (("SINGLE_SEASON",ref),("SUPER_LEAGUE",other)):
            tr=frame.iloc[tr_idx]; va=frame.iloc[va_idx]
            if set(tr["outcome"].astype(str)) != set(CLASS_ORDER):
                # This is a diagnostic, so record rather than silently changing chronology.
                rows.append({"fold":fold,"representation":name,"train_n":len(tr),"validation_n":len(va),"rps":np.nan,"status":"skipped_missing_train_class"})
                continue
            model=make_rf_pipeline(cfg)
            model.fit(tr[list(feature_cols)],tr["outcome"].astype(str))
            probs=aligned_predict_proba(model,va[list(feature_cols)])
            rows.append({"fold":fold,"representation":name,"train_n":len(tr),"validation_n":len(va),"rps":float(per_match_rps(va["outcome"].astype(str),probs).mean()),"status":"ok"})
    return pd.DataFrame(rows)


def coverage_by_matchweek(target: pd.DataFrame, coverage_detail: pd.DataFrame) -> pd.DataFrame:
    if "match_week" not in target.columns:
        return pd.DataFrame()
    week=target[["match_id","match_week"]].copy()
    week["match_week"]=pd.to_numeric(week["match_week"],errors="coerce")
    m=coverage_detail.merge(week,on="match_id",how="left",validate="one_to_one")
    m=m.dropna(subset=["match_week"])
    if m.empty: return pd.DataFrame()
    out=(m.groupby("match_week",as_index=False)
         .agg(matches=("match_id","count"),
              single_eligible=("single_eligible","sum"),
              superleague_eligible=("superleague_eligible","sum"),
              common_eligible=("common_eligible","sum")))
    out["single_coverage_pct"]=100*out["single_eligible"]/out["matches"]
    out["superleague_coverage_pct"]=100*out["superleague_eligible"]/out["matches"]
    return out


def plot_coverage_by_week(week: pd.DataFrame,path: Path)->None:
    if week.empty: return
    fig,ax=plt.subplots(figsize=(9,4.8))
    ax.plot(week["match_week"],week["single_coverage_pct"],marker="o",label="Single-season")
    ax.plot(week["match_week"],week["superleague_coverage_pct"],marker="o",label="Super League")
    ax.set_xlabel("Match week")
    ax.set_ylabel("Eligible matches (%)")
    ax.set_ylim(-2,102)
    ax.set_title("Berrar eligibility: season reset vs cross-season Super League")
    ax.legend()
    fig.tight_layout()
    path.parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(path,dpi=160)
    plt.close(fig)


def plot_rps(metrics: pd.DataFrame,path: Path,title: str)->None:
    m=metrics.sort_values("rps",ascending=True)
    fig,ax=plt.subplots(figsize=(7,4.2))
    ax.barh(m["representation"],m["rps"])
    ax.set_xlabel("RPS (lower is better)")
    ax.set_title(title)
    fig.tight_layout()
    path.parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(path,dpi=160)
    plt.close(fig)


def environment_manifest()->dict[str,Any]:
    return {
        "python":sys.version,
        "platform":platform.platform(),
        "numpy":np.__version__,
        "pandas":pd.__version__,
        "sklearn":sklearn.__version__,
    }


## 5. Load frozen upstream audits, hashes, and EXP01 probe configuration

In [5]:

import json
import pandas as pd
from IPython.display import display

def require_passed_audit(path: Path, label: str) -> pd.DataFrame:
    audit = pd.read_csv(path)
    if "passed" not in audit.columns:
        raise KeyError(f"{label} audit has no passed column: {path}")
    passed = as_bool_series(audit["passed"])
    if not bool(passed.all()):
        display(audit.loc[~passed])
        raise AssertionError(f"{label} contains failed checks; EXP01B is blocked.")
    print(f"{label}: {int(passed.sum())}/{len(passed)} PASS")
    return audit

p1_ready = require_passed_audit(P1_READINESS, "P1 readiness")
p1_leak = require_passed_audit(P1_LEAKAGE, "P1 leakage/source tests")
exp01_ready = require_passed_audit(EXP01_READINESS, "EXP01 readiness")

p1_summary = json.loads(P1_SUMMARY.read_text(encoding="utf-8"))
p1_selected_record = json.loads(P1_SELECTED.read_text(encoding="utf-8"))
exp01_freeze = json.loads(EXP01_FREEZE.read_text(encoding="utf-8"))

selected_n = int(p1_summary["selected_n_pearson"])
if selected_n != EXPECTED_SELECTED_N:
    raise AssertionError(
        f"Frozen P1 n changed: expected {EXPECTED_SELECTED_N}, observed {selected_n}"
    )
if int(exp01_freeze["selected_n"]) != selected_n:
    raise AssertionError("EXP01 freeze record and P1 run disagree on selected n")

probe_kwargs = {
    k: v
    for k, v in exp01_freeze["rf_probe"].items()
    if k in ProbeConfig.__dataclass_fields__
}
probe_kwargs["temporal_splits"] = TEMPORAL_FOLDS
cfg = ProbeConfig(**probe_kwargs)

input_manifest = pd.DataFrame([
    {
        "name": name,
        "path": str(path),
        "bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }
    for name, path in REQUIRED_FILES.items()
])
input_manifest.to_csv(EXPERIMENT_ROOT / "audit" / "input_manifest.csv", index=False)

env = environment_manifest()
env["selected_n"] = selected_n
env["paper_total_features"] = list(PAPER_TOTAL_FEATURES)
env["probe_config_from_exp01"] = asdict(cfg)
write_json(env, EXPERIMENT_ROOT / "audit" / "environment_and_frozen_config.json")

print("Frozen selected n:", selected_n)
print("RF probe loaded from EXP01 pre-test freeze:")
display(pd.Series(asdict(cfg), name="value").to_frame())


P1 readiness: 15/15 PASS
P1 leakage/source tests: 13/13 PASS
EXP01 readiness: 10/10 PASS
Frozen selected n: 28
RF probe loaded from EXP01 pre-test freeze:


,value
n_estimators,500
max_depth,None
min_samples_leaf,3
max_features,sqrt
class_weight,None
random_state,42
n_jobs,-1
imputer_strategy,median
ece_bins,10
bootstrap_resamples,10000



## 6. Rebuild the **single-season** Berrar representation

This is the only new feature-generation step.

The historical index is deliberately constructed from **StatsBomb EPL 2015/16 target matches only**. Therefore the first match of the season has zero prior same-season history, and prior seasons cannot contribute.

The Super-League arm is **not rebuilt**; it is loaded from the already verified P1 output.


In [6]:

target = pd.read_parquet(P1_TARGET).sort_values(["kickoff", "match_id"]).reset_index(drop=True)
superleague = pd.read_parquet(P1_SUPER).sort_values(["kickoff", "match_id"]).reset_index(drop=True)
p1_split = pd.read_csv(P1_SPLIT)

if len(target) != 380 or target["match_id"].nunique() != 380:
    raise AssertionError(f"Expected 380 StatsBomb target matches; got rows={len(target)}, unique={target['match_id'].nunique()}")
if len(superleague) != 380 or superleague["match_id"].nunique() != 380:
    raise AssertionError("Frozen Super-League table is not the expected 380-match target table")

_, p1_cfg = build_p1_default_config(
    results_root=EXPERIMENT_ROOT / "_config_only",
    run_mode="real",
    run_name="EXP01B_CONFIG_ONLY",
    full_recency_search=True,
)

if int(p1_cfg.p1_min_prior_matches) != 6:
    raise AssertionError(f"Expected Berrar minimum history 6; observed {p1_cfg.p1_min_prior_matches}")

split_map = {
    int(r["match_id"]): str(r["split"])
    for r in p1_split[["match_id", "split"]].to_dict("records")
}

single_team_facts = build_p1_team_match_facts(target)
single_history_index = P1HistoryIndex(single_team_facts)
current_team_sets = build_p1_current_team_sets(target)

single_season = build_p1_features_for_n(
    p1_cfg,
    target,
    single_history_index,
    current_team_sets,
    split_map,
    selected_n,
    include_homeaway=False,
)
single_season = single_season.sort_values(["kickoff", "match_id"]).reset_index(drop=True)

assert_no_p1_prematch_leakage(single_season)

single_team_facts.to_parquet(
    EXPERIMENT_ROOT / "data" / "single_season_team_match_facts.parquet",
    index=False,
)
single_season.to_parquet(
    EXPERIMENT_ROOT / "data" / "single_season_features_n28.parquet",
    index=False,
)

print("Single-season Berrar features generated.")
print("Rows:", len(single_season))
print("Eligible:", int(as_bool_series(single_season["p1_eligible"]).sum()))
print("Frozen Super-League eligible:", int(as_bool_series(superleague["p1_eligible"]).sum()))


Single-season Berrar features generated.
Rows: 380
Eligible: 320
Frozen Super-League eligible: 374


## 7. Methodology and leakage checks for the new arm

In [7]:

check_rows = []

def add_check(name: str, passed: bool, details: str):
    check_rows.append({"check": name, "passed": bool(passed), "details": details})

# Exact target and feature contracts.
add_check(
    "target IDs identical",
    set(single_season["match_id"].astype(int)) == set(superleague["match_id"].astype(int)),
    f"single={single_season['match_id'].nunique()}, super={superleague['match_id'].nunique()}",
)
add_check(
    "six total feature contract",
    all(c in single_season.columns and c in superleague.columns for c in PAPER_TOTAL_FEATURES),
    f"feature_count={len(PAPER_TOTAL_FEATURES)}",
)
add_check(
    "recency frozen at selected n",
    sorted(pd.to_numeric(single_season["recency_n"], errors="coerce").dropna().astype(int).unique().tolist()) == [selected_n]
    and sorted(pd.to_numeric(superleague["recency_n"], errors="coerce").dropna().astype(int).unique().tolist()) == [selected_n],
    f"n={selected_n}",
)

# Single-season source restriction.
fact_seasons = sorted(single_team_facts["season_name"].astype(str).unique().tolist())
add_check(
    "single-season history contains target season only",
    fact_seasons == [str(p1_cfg.season_name)],
    f"fact_seasons={fact_seasons}",
)
add_check(
    "single-season facts contain no historical synthetic IDs",
    bool((pd.to_numeric(single_team_facts["match_id"], errors="coerce") > 0).all()),
    "all source match IDs are StatsBomb target-season match IDs",
)

# Season reset: first chronological target has no prior current-season total history.
first = single_season.sort_values(["kickoff", "match_id"]).iloc[0]
add_check(
    "history resets at season start",
    int(first["home_total_history_available"]) == 0
    and int(first["away_total_history_available"]) == 0,
    f"first_match={int(first['match_id'])}, home_prior={int(first['home_total_history_available'])}, away_prior={int(first['away_total_history_available'])}",
)

# Minimum-six rule.
expected_eligible = (
    (pd.to_numeric(single_season["home_total_history_available"]) >= int(p1_cfg.p1_min_prior_matches))
    & (pd.to_numeric(single_season["away_total_history_available"]) >= int(p1_cfg.p1_min_prior_matches))
)
observed_eligible = as_bool_series(single_season["p1_eligible"])
add_check(
    "single-season minimum-six eligibility",
    bool((expected_eligible == observed_eligible).all()),
    f"minimum={p1_cfg.p1_min_prior_matches}",
)

# Super League should not lose eligibility relative to a strict season reset.
single_ids = set(single_season.loc[observed_eligible, "match_id"].astype(int))
super_ids = set(superleague.loc[as_bool_series(superleague["p1_eligible"]), "match_id"].astype(int))
add_check(
    "Super League eligibility is a superset",
    single_ids.issubset(super_ids),
    f"single_eligible={len(single_ids)}, super_eligible={len(super_ids)}",
)

# Split alignment.
split_compare = single_season[["match_id","split"]].merge(
    superleague[["match_id","split"]],
    on="match_id",
    suffixes=("_single","_super"),
    validate="one_to_one",
)
add_check(
    "split alignment",
    bool((split_compare["split_single"].astype(str) == split_compare["split_super"].astype(str)).all()),
    f"mismatches={int((split_compare['split_single'].astype(str) != split_compare['split_super'].astype(str)).sum())}",
)

def rebuild_single(mut_target: pd.DataFrame) -> pd.DataFrame:
    facts = build_p1_team_match_facts(mut_target)
    idx = P1HistoryIndex(facts)
    members = build_p1_current_team_sets(mut_target)
    rebuilt = build_p1_features_for_n(
        p1_cfg, mut_target, idx, members, split_map, selected_n, include_homeaway=False
    )
    assert_no_p1_prematch_leakage(rebuilt)
    return rebuilt.sort_values(["kickoff","match_id"]).reset_index(drop=True)

# Target-result perturbation.
eligible_rows = single_season[observed_eligible].sort_values(["kickoff","match_id"])
if eligible_rows.empty:
    raise AssertionError("No single-season eligible row for perturbation tests")

target_probe_id = int(eligible_rows.iloc[len(eligible_rows)//2]["match_id"])
mut = target.copy()
mask = mut["match_id"].astype(int) == target_probe_id
mut.loc[mask, ["home_score","away_score","goal_margin_raw","goal_margin_clipped"]] = [999, 998, 1, 1]
mut.loc[mask, "outcome"] = "H"
rebuilt = rebuild_single(mut)

before = single_season.set_index("match_id").loc[target_probe_id]
after = rebuilt.set_index("match_id").loc[target_probe_id]
add_check(
    "target-result perturbation",
    feature_values_equal(before, after, PAPER_TOTAL_FEATURES),
    f"target={target_probe_id} own result cannot alter own pre-match vector",
)

# Future-result perturbation.
early = eligible_rows.iloc[max(0, len(eligible_rows)//4)]
early_id = int(early["match_id"])
later = target[pd.to_datetime(target["kickoff"]) > pd.Timestamp(early["kickoff"])].sort_values(["kickoff","match_id"])
if later.empty:
    raise AssertionError("No future row available for perturbation test")
future_id = int(later.iloc[-1]["match_id"])
mut2 = target.copy()
mask2 = mut2["match_id"].astype(int) == future_id
mut2.loc[mask2, ["home_score","away_score","goal_margin_raw","goal_margin_clipped"]] = [99, 0, 99, 5]
mut2.loc[mask2, "outcome"] = "H"
rebuilt2 = rebuild_single(mut2)
before2 = single_season.set_index("match_id").loc[early_id]
after2 = rebuilt2.set_index("match_id").loc[early_id]
add_check(
    "future-result perturbation",
    feature_values_equal(before2, after2, PAPER_TOTAL_FEATURES),
    f"future={future_id} cannot alter earlier={early_id}",
)

method_checks = pd.DataFrame(check_rows)
method_checks.to_csv(
    EXPERIMENT_ROOT / "audit" / "exp01b_leakage_and_method_checks.csv",
    index=False,
)

display(method_checks)
if not bool(method_checks["passed"].all()):
    display(method_checks.loc[~method_checks["passed"]])
    raise AssertionError("EXP01B methodology/leakage checks failed.")

print("EXP01B methodology/leakage checks: PASS")


,check,passed,details
0,target IDs identical,True,"single=380, super=380"
1,six total feature contract,True,feature_count=6
2,recency frozen at selected n,True,n=28
3,single-season history contains target season only,True,fact_seasons=['2015/2016']
4,single-season facts contain no historical synt...,True,all source match IDs are StatsBomb target-seas...
5,history resets at season start,True,"first_match=3754097, home_prior=0, away_prior=0"
6,single-season minimum-six eligibility,True,minimum=6
7,Super League eligibility is a superset,True,"single_eligible=320, super_eligible=374"
8,split alignment,True,mismatches=0
9,target-result perturbation,True,target=3754348 own result cannot alter own pre...


EXP01B methodology/leakage checks: PASS


## 8. Coverage and cold-start comparison

In [8]:

coverage_detail, coverage_summary = build_coverage_audit(single_season, superleague)

coverage_detail.to_csv(
    EXPERIMENT_ROOT / "data" / "coverage_match_level.csv",
    index=False,
)
coverage_summary.to_csv(
    EXPERIMENT_ROOT / "metrics" / "coverage_comparison.csv",
    index=False,
)

week_coverage = coverage_by_matchweek(target, coverage_detail)
if not week_coverage.empty:
    week_coverage.to_csv(
        EXPERIMENT_ROOT / "metrics" / "coverage_by_matchweek.csv",
        index=False,
    )
    plot_coverage_by_week(
        week_coverage,
        EXPERIMENT_ROOT / "plots" / "eligibility_by_matchweek.png",
    )

print("Coverage comparison")
display(coverage_summary)

all_row = coverage_summary.set_index("split").loc["all"]
coverage_gain = int(all_row["superleague_additional_eligible"])
print(f"Super League adds {coverage_gain} eligible target matches relative to season reset.")

if not week_coverage.empty:
    print("Coverage by StatsBomb match week")
    display(week_coverage)


Coverage comparison


,split,total_matches,single_eligible,superleague_eligible,common_eligible,superleague_additional_eligible,single_coverage_pct,superleague_coverage_pct
0,all,380,320,374,320,54,84.210526,98.421053
1,train,257,197,251,197,54,76.653696,97.665370
2,validation,77,77,77,77,0,100.000000,100.000000
3,test,46,46,46,46,0,100.000000,100.000000


Super League adds 54 eligible target matches relative to season reset.
Coverage by StatsBomb match week


,match_week,matches,single_eligible,superleague_eligible,common_eligible,single_coverage_pct,superleague_coverage_pct
0,1,10,0,9,0,0.0,90.0
1,2,10,0,9,0,0.0,90.0
2,3,10,0,9,0,0.0,90.0
3,4,10,0,9,0,0.0,90.0
4,5,10,0,9,0,0.0,90.0
5,6,10,0,9,0,0.0,90.0
6,7,10,10,10,10,100.0,100.0
7,8,10,10,10,10,100.0,100.0
8,9,10,10,10,10,100.0,100.0
9,10,10,10,10,10,100.0,100.0



## 9. Build the exact common comparison population

Predictive performance is evaluated **only where both arms are eligible**.

This prevents the Super-League arm from receiving credit merely because it covers more matches. Coverage and predictive quality are reported separately.


In [9]:

tables = build_common_feature_tables(
    single=single_season,
    superleague=superleague,
    feature_cols=PAPER_TOTAL_FEATURES,
)

common_manifest = tables["SINGLE_SEASON"][
    ["match_id","split","kickoff","home_team_id","away_team_id","outcome","goal_margin_raw"]
].copy()
common_manifest.to_csv(
    EXPERIMENT_ROOT / "data" / "common_match_manifest.csv",
    index=False,
)

tables["SINGLE_SEASON"].to_parquet(
    EXPERIMENT_ROOT / "data" / "single_season_common.parquet",
    index=False,
)
tables["SUPER_LEAGUE"].to_parquet(
    EXPERIMENT_ROOT / "data" / "superleague_common.parquet",
    index=False,
)

common_counts = common_manifest["split"].value_counts().to_dict()
if any(common_counts.get(s, 0) <= 0 for s in ("train","validation","test")):
    raise AssertionError(f"Common population must include all three splits; observed {common_counts}")

print("Common population:", len(common_manifest))
print("Common split counts:", common_counts)
print("Both arms use exactly the same match IDs and six feature definitions: PASS")


Common population: 320
Common split counts: {'train': 197, 'validation': 77, 'test': 46}
Both arms use exactly the same match IDs and six feature definitions: PASS


## 10. Train-only expanding temporal diagnostic

In [10]:

fold_metrics = temporal_fold_probe(
    tables=tables,
    feature_cols=PAPER_TOTAL_FEATURES,
    cfg=cfg,
)
fold_metrics.to_csv(
    EXPERIMENT_ROOT / "metrics" / "temporal_fold_metrics.csv",
    index=False,
)

display(fold_metrics)

fold_ok = fold_metrics[fold_metrics["status"] == "ok"].copy()
if not fold_ok.empty:
    fold_pivot = fold_ok.pivot(index="fold", columns="representation", values="rps")
    if {"SINGLE_SEASON","SUPER_LEAGUE"}.issubset(fold_pivot.columns):
        fold_pivot["delta_super_minus_single"] = (
            fold_pivot["SUPER_LEAGUE"] - fold_pivot["SINGLE_SEASON"]
        )
        fold_pivot.to_csv(
            EXPERIMENT_ROOT / "metrics" / "temporal_fold_rps_deltas.csv"
        )
        print("Train-only temporal fold deltas; negative favors Super League")
        display(fold_pivot)


,fold,representation,train_n,validation_n,rps,status
0,1,SINGLE_SEASON,50,49,0.251246,ok
1,1,SUPER_LEAGUE,50,49,0.264287,ok
2,2,SINGLE_SEASON,99,49,0.233791,ok
3,2,SUPER_LEAGUE,99,49,0.232930,ok
4,3,SINGLE_SEASON,148,49,0.235111,ok
5,3,SUPER_LEAGUE,148,49,0.221676,ok


Train-only temporal fold deltas; negative favors Super League


representation,SINGLE_SEASON,SUPER_LEAGUE,delta_super_minus_single
fold,,,
1,0.251246,0.264287,0.013041
2,0.233791,0.232930,-0.000861
3,0.235111,0.221676,-0.013435


## 11. Fit the same frozen RF probe on the common training population

In [11]:

models = fit_pair(
    tables=tables,
    feature_cols=PAPER_TOTAL_FEATURES,
    cfg=cfg,
    models_dir=EXPERIMENT_ROOT / "models",
)

common_train_n = int((common_manifest["split"] == "train").sum())
print("Common training matches:", common_train_n)
print("Models fitted with identical EXP01-frozen RF probe: PASS")


Common training matches: 197
Models fitted with identical EXP01-frozen RF probe: PASS


## 12. Validation comparison

In [12]:

val_predictions, val_metrics = evaluate_pair(
    models=models,
    tables=tables,
    feature_cols=PAPER_TOTAL_FEATURES,
    cfg=cfg,
    split="validation",
)
val_bootstrap = paired_rps_bootstrap(
    val_predictions,
    n_resamples=cfg.bootstrap_resamples,
    seed=cfg.random_state,
)

val_predictions.to_csv(
    EXPERIMENT_ROOT / "predictions" / "validation_predictions.csv",
    index=False,
)
val_metrics.to_csv(
    EXPERIMENT_ROOT / "metrics" / "validation_metrics.csv",
    index=False,
)
val_bootstrap.to_csv(
    EXPERIMENT_ROOT / "metrics" / "validation_paired_rps_bootstrap.csv",
    index=False,
)

print("VALIDATION — same matches, same learner, same six features, same n")
display(val_metrics)
print("Paired bootstrap: RPS(Super League) - RPS(Single Season); negative favors Super League")
display(val_bootstrap)


VALIDATION — same matches, same learner, same six features, same n


,representation,n_matches,rps,log_loss,brier,accuracy,ece,feature_count,model
0,SUPER_LEAGUE,77,0.210120,1.01550,0.606529,0.467532,0.108221,6,RandomForest fixed EXP01 probe
1,SINGLE_SEASON,77,0.219871,1.05825,0.636983,0.493506,0.081209,6,RandomForest fixed EXP01 probe


Paired bootstrap: RPS(Super League) - RPS(Single Season); negative favors Super League


,left,right,n_matches,delta_definition,mean_delta_rps,ci95_low,ci95_high,bootstrap_resamples,bootstrap_seed,fraction_bootstrap_delta_lt_0
0,SUPER_LEAGUE,SINGLE_SEASON,77,RPS(SUPER_LEAGUE) - RPS(SINGLE_SEASON); negati...,-0.009751,-0.027929,0.007217,10000,42,0.8669



## 13. Freeze EXP01B before report-only test evaluation

The test set has already been exposed in the preceding project workflow, so EXP01B is explicitly treated as a **diagnostic P1 ablation**, not a new model-selection opportunity.

This record prevents any change to:
- `n=28`;
- six-feature total representation;
- eligibility rule;
- common population construction;
- RF probe configuration;
- validation analysis

after the test results are displayed.


In [13]:

freeze_record = {
    "experiment": "EXP01B_SUPERLEAGUE_ABLATION",
    "purpose": "diagnostic isolation of cross-season Super-League history",
    "test_status_at_write": "NOT_EVALUATED_IN_EXP01B_YET",
    "selected_n": selected_n,
    "recency_policy": "same frozen n for both arms; no single-season re-selection",
    "feature_family": "Berrar total",
    "feature_count_each_arm": len(PAPER_TOTAL_FEATURES),
    "paper_total_features": list(PAPER_TOTAL_FEATURES),
    "minimum_prior_matches": int(p1_cfg.p1_min_prior_matches),
    "single_season_history_source": "StatsBomb EPL 2015/16 completed earlier target matches only",
    "superleague_history_source": "frozen TA-approved P1 Super-League output",
    "common_population_n": len(common_manifest),
    "common_split_counts": common_counts,
    "coverage_summary": coverage_summary.to_dict("records"),
    "rf_probe_from_exp01_freeze": asdict(cfg),
    "validation_metrics": val_metrics.to_dict("records"),
    "validation_bootstrap": val_bootstrap.to_dict("records"),
    "temporal_fold_metrics": fold_metrics.to_dict("records"),
    "input_manifest": input_manifest.to_dict("records"),
}
freeze_path = EXPERIMENT_ROOT / "audit" / "FROZEN_BEFORE_TEST.json"
write_json(freeze_record, freeze_path)

if not freeze_path.exists():
    raise AssertionError("EXP01B pre-test freeze record was not written.")

print("EXP01B frozen before report-only test:", freeze_path)


EXP01B frozen before report-only test: /content/drive/MyDrive/ml_project/code/data_pipeline/results/EXP01B_SUPERLEAGUE_ABLATION/audit/FROZEN_BEFORE_TEST.json


## 14. Test evaluation — report only

In [14]:

test_predictions = None
test_metrics = None
test_bootstrap = None

if RUN_TEST_STAGE:
    if not (EXPERIMENT_ROOT / "audit" / "FROZEN_BEFORE_TEST.json").exists():
        raise AssertionError("Missing EXP01B freeze record; test stage blocked.")

    test_predictions, test_metrics = evaluate_pair(
        models=models,
        tables=tables,
        feature_cols=PAPER_TOTAL_FEATURES,
        cfg=cfg,
        split="test",
    )
    test_bootstrap = paired_rps_bootstrap(
        test_predictions,
        n_resamples=cfg.bootstrap_resamples,
        seed=cfg.random_state,
    )

    test_predictions.to_csv(
        EXPERIMENT_ROOT / "predictions" / "test_predictions.csv",
        index=False,
    )
    test_metrics.to_csv(
        EXPERIMENT_ROOT / "metrics" / "test_metrics.csv",
        index=False,
    )
    test_bootstrap.to_csv(
        EXPERIMENT_ROOT / "metrics" / "test_paired_rps_bootstrap.csv",
        index=False,
    )

    print("TEST — report only")
    display(test_metrics)
    print("Paired bootstrap: RPS(Super League) - RPS(Single Season)")
    display(test_bootstrap)
else:
    print("RUN_TEST_STAGE=False; EXP01B remains frozen after validation.")


TEST — report only


,representation,n_matches,rps,log_loss,brier,accuracy,ece,feature_count,model
0,SINGLE_SEASON,46,0.198972,1.031246,0.614843,0.500000,0.079055,6,RandomForest fixed EXP01 probe
1,SUPER_LEAGUE,46,0.215653,1.076355,0.648008,0.456522,0.106663,6,RandomForest fixed EXP01 probe


Paired bootstrap: RPS(Super League) - RPS(Single Season)


,left,right,n_matches,delta_definition,mean_delta_rps,ci95_low,ci95_high,bootstrap_resamples,bootstrap_seed,fraction_bootstrap_delta_lt_0
0,SUPER_LEAGUE,SINGLE_SEASON,46,RPS(SUPER_LEAGUE) - RPS(SINGLE_SEASON); negati...,0.016681,-0.00233,0.036725,10000,42,0.0424


## 15. Plots

In [15]:

plot_rps(
    val_metrics,
    EXPERIMENT_ROOT / "plots" / "validation_rps_single_vs_superleague.png",
    title="EXP01B validation RPS",
)

if RUN_TEST_STAGE and test_metrics is not None:
    plot_rps(
        test_metrics,
        EXPERIMENT_ROOT / "plots" / "test_rps_single_vs_superleague.png",
        title="EXP01B test RPS — report only",
    )

print("Plots written to:", EXPERIMENT_ROOT / "plots")


Plots written to: /content/drive/MyDrive/ml_project/code/data_pipeline/results/EXP01B_SUPERLEAGUE_ABLATION/plots


## 16. Final EXP01B readiness and data-stage closure

In [16]:

def metric_value(frame: pd.DataFrame, representation: str, column: str) -> float:
    return float(frame.set_index("representation").loc[representation, column])

val_delta = (
    metric_value(val_metrics, "SUPER_LEAGUE", "rps")
    - metric_value(val_metrics, "SINGLE_SEASON", "rps")
)

test_delta = None
if RUN_TEST_STAGE and test_metrics is not None:
    test_delta = (
        metric_value(test_metrics, "SUPER_LEAGUE", "rps")
        - metric_value(test_metrics, "SINGLE_SEASON", "rps")
    )

readiness_rows = [
    {
        "check": "upstream P1 readiness",
        "passed": bool(as_bool_series(p1_ready["passed"]).all()),
        "details": f"{int(as_bool_series(p1_ready['passed']).sum())}/{len(p1_ready)}",
    },
    {
        "check": "upstream EXP01 readiness",
        "passed": bool(as_bool_series(exp01_ready["passed"]).all()),
        "details": f"{int(as_bool_series(exp01_ready['passed']).sum())}/{len(exp01_ready)}",
    },
    {
        "check": "selected n frozen",
        "passed": selected_n == EXPECTED_SELECTED_N,
        "details": f"n={selected_n}",
    },
    {
        "check": "single-season causal method checks",
        "passed": bool(method_checks["passed"].all()),
        "details": f"{int(method_checks['passed'].sum())}/{len(method_checks)}",
    },
    {
        "check": "Super League eligibility superset",
        "passed": single_ids.issubset(super_ids),
        "details": f"single={len(single_ids)}, super={len(super_ids)}",
    },
    {
        "check": "common train/validation/test population exists",
        "passed": all(common_counts.get(s, 0) > 0 for s in ("train","validation","test")),
        "details": str(common_counts),
    },
    {
        "check": "same six features in both predictive arms",
        "passed": len(PAPER_TOTAL_FEATURES) == 6,
        "details": str(list(PAPER_TOTAL_FEATURES)),
    },
    {
        "check": "train-only temporal diagnostic generated",
        "passed": not fold_metrics.empty,
        "details": f"rows={len(fold_metrics)}",
    },
    {
        "check": "validation comparison generated",
        "passed": len(val_metrics) == 2 and len(val_bootstrap) == 1,
        "details": f"metrics={len(val_metrics)}, bootstrap={len(val_bootstrap)}",
    },
    {
        "check": "freeze written before EXP01B test",
        "passed": freeze_path.exists(),
        "details": str(freeze_path),
    },
]

if RUN_TEST_STAGE:
    readiness_rows.append({
        "check": "report-only test comparison generated",
        "passed": test_metrics is not None and len(test_metrics) == 2 and test_bootstrap is not None and len(test_bootstrap) == 1,
        "details": "two matched arms + one paired bootstrap",
    })

readiness = pd.DataFrame(readiness_rows)
readiness.to_csv(
    EXPERIMENT_ROOT / "audit" / "exp01b_readiness.csv",
    index=False,
)

if not bool(readiness["passed"].all()):
    display(readiness.loc[~readiness["passed"]])
    raise AssertionError("EXP01B readiness failed; data stage cannot be closed.")

summary = {
    "experiment": "EXP01B_SUPERLEAGUE_ABLATION",
    "status": "PASS",
    "selected_n": selected_n,
    "feature_count": len(PAPER_TOTAL_FEATURES),
    "single_season_eligible": int(all_row["single_eligible"]),
    "superleague_eligible": int(all_row["superleague_eligible"]),
    "superleague_additional_eligible": coverage_gain,
    "common_matches": len(common_manifest),
    "common_split_counts": common_counts,
    "validation_rps_super_minus_single": val_delta,
    "validation_metrics": val_metrics.to_dict("records"),
    "validation_bootstrap": val_bootstrap.to_dict("records"),
    "test_stage_run": bool(RUN_TEST_STAGE),
    "test_rps_super_minus_single": test_delta,
    "test_metrics": test_metrics.to_dict("records") if test_metrics is not None else None,
    "test_bootstrap": test_bootstrap.to_dict("records") if test_bootstrap is not None else None,
    "interpretation_rule": {
        "coverage": "positive superleague_additional_eligible means cross-season history reduces cold start",
        "rps_delta": "negative RPS(Super League)-RPS(Single Season) favors Super League",
        "test": "report-only; must not change n or upstream representation decisions",
    },
}
write_json(summary, EXPERIMENT_ROOT / "audit" / "experiment_summary.json")

closure = {
    "status": "DATA_STAGE_EXPERIMENTS_COMPLETE",
    "condition": "EXP01B readiness passed",
    "immutable_decisions": {
        "P1_selected_n": selected_n,
        "P1_min_prior_matches": int(p1_cfg.p1_min_prior_matches),
        "P1_feature_contracts": "6 total / 18 homeaway",
        "outer_split": "existing chronological match-level split",
        "bookmaker_role": "independent market baseline",
    },
    "completed_data_experiments": [
        "MD1 causal StatsBomb pipeline",
        "Berrar P1 Super-League construction and recency selection",
        "EXP01 representation ablation",
        "EXP01B single-season vs Super-League ablation",
    ],
    "next_phase": "prediction/modeling: full Task C model suite, then Task R and Task L",
    "rule": "Do not change P1 n, target split, feature definitions, or eligibility rules based on later test/model results.",
}
write_json(closure, EXPERIMENT_ROOT / "audit" / "DATA_STAGE_CLOSURE.json")

readme = f"""# EXP01B — Single-Season vs Super-League Berrar Ablation

Status: PASS

- Frozen Berrar recency: n={selected_n}
- Feature family: six `total` Berrar features
- Single-season eligible matches: {int(all_row['single_eligible'])}
- Super-League eligible matches: {int(all_row['superleague_eligible'])}
- Additional matches made eligible by cross-season history: {coverage_gain}
- Common predictive population: {len(common_manifest)}
- Common split counts: {common_counts}
- Validation RPS delta, Super League - Single Season: {val_delta:.8f}
- Test RPS delta, Super League - Single Season: {test_delta if test_delta is not None else 'not run'}

Negative RPS delta favors the Super-League representation.

The test stage is report-only. The experiment does not re-select recency or alter
the feature definition. See `audit/FROZEN_BEFORE_TEST.json` and
`audit/DATA_STAGE_CLOSURE.json`.
"""
(EXPERIMENT_ROOT / "README.md").write_text(readme, encoding="utf-8")

print("EXP01B READINESS")
display(readiness)

print("FINAL COVERAGE")
display(coverage_summary)

print("VALIDATION METRICS")
display(val_metrics)

if RUN_TEST_STAGE and test_metrics is not None:
    print("TEST METRICS — REPORT ONLY")
    display(test_metrics)

print()
print("DATA-SIDE EXPERIMENTS COMPLETE.")
print("Next phase: full prediction/modeling experiments.")
print("EXP01B output:", EXPERIMENT_ROOT)


EXP01B READINESS


,check,passed,details
0,upstream P1 readiness,True,15/15
1,upstream EXP01 readiness,True,10/10
2,selected n frozen,True,n=28
3,single-season causal method checks,True,11/11
4,Super League eligibility superset,True,"single=320, super=374"
5,common train/validation/test population exists,True,"{'train': 197, 'validation': 77, 'test': 46}"
6,same six features in both predictive arms,True,"['p1_home_scr_total', 'p1_home_con_total', 'p1..."
7,train-only temporal diagnostic generated,True,rows=6
8,validation comparison generated,True,"metrics=2, bootstrap=1"
9,freeze written before EXP01B test,True,/content/drive/MyDrive/ml_project/code/data_pi...


FINAL COVERAGE


,split,total_matches,single_eligible,superleague_eligible,common_eligible,superleague_additional_eligible,single_coverage_pct,superleague_coverage_pct
0,all,380,320,374,320,54,84.210526,98.421053
1,train,257,197,251,197,54,76.653696,97.665370
2,validation,77,77,77,77,0,100.000000,100.000000
3,test,46,46,46,46,0,100.000000,100.000000


VALIDATION METRICS


,representation,n_matches,rps,log_loss,brier,accuracy,ece,feature_count,model
0,SUPER_LEAGUE,77,0.210120,1.01550,0.606529,0.467532,0.108221,6,RandomForest fixed EXP01 probe
1,SINGLE_SEASON,77,0.219871,1.05825,0.636983,0.493506,0.081209,6,RandomForest fixed EXP01 probe


TEST METRICS — REPORT ONLY


,representation,n_matches,rps,log_loss,brier,accuracy,ece,feature_count,model
0,SINGLE_SEASON,46,0.198972,1.031246,0.614843,0.500000,0.079055,6,RandomForest fixed EXP01 probe
1,SUPER_LEAGUE,46,0.215653,1.076355,0.648008,0.456522,0.106663,6,RandomForest fixed EXP01 probe



DATA-SIDE EXPERIMENTS COMPLETE.
Next phase: full prediction/modeling experiments.
EXP01B output: /content/drive/MyDrive/ml_project/code/data_pipeline/results/EXP01B_SUPERLEAGUE_ABLATION



## Interpretation guide

After the real run, interpret **coverage** and **predictive quality** separately.

- If the Super League adds eligible matches, it has reduced the beginning-of-season cold-start problem.
- If `RPS(Super League) - RPS(Single Season) < 0` on validation, cross-season history also improved predictive quality on the matched population.
- If coverage improves but RPS does not, the correct conclusion is that Super-League history improves **data availability** but did not establish a predictive advantage on EPL 2015/16.
- If neither improves, report the negative result directly.

Do not re-select `n` or create a new P1 variant after seeing the test result.
